# 01 · Tyre Degradation Analysis

Fit per-compound degradation curves from real race laps (OpenF1 API).

**Pipeline:** load race → keep clean green-flag laps → remove fuel-burn effect → pool drivers via per-driver baseline → fit linear vs quadratic per compound (5-fold CV) → detect cliff → save models for `02_pit_strategy`.

In [ ]:
import sys, os
if "google.colab" in sys.modules and not os.path.exists("f1strat"):
    REPO = "https://github.com/anerudh-k/f1-tyre-strategy"
    !git clone -q {REPO} repo && cp -r repo/* .
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from f1strat import *
COLORS = {"SOFT": "#e10600", "MEDIUM": "#f5c400", "HARD": "#8a8a8a"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
os.makedirs("figures", exist_ok=True); os.makedirs("data", exist_ok=True)

In [ ]:
YEAR, GP = 2025, "Bahrain"
DEMO = False
FUEL_START_KG = 100.0
SEC_PER_KG = 0.03

## 1. Load and clean

In [ ]:
if DEMO:
    laps_raw, TOTAL_LAPS, EVENT = make_synthetic_laps(), 57, "Synthetic GP"
else:
    session, laps_raw = load_race(YEAR, GP)
    TOTAL_LAPS, EVENT = int(laps_raw["LapNumber"].max()), session.event["EventName"]

clean = clean_laps(laps_raw)
df = fuel_correct(clean, TOTAL_LAPS, FUEL_START_KG, SEC_PER_KG)
print(f"{EVENT} {YEAR}: {TOTAL_LAPS} laps | {len(laps_raw)} raw laps -> {len(df)} clean")
df.groupby("Compound").agg(laps=("LapTimeS", "size"), drivers=("Driver", "nunique"),
                           max_age=("TyreLife", "max"))

## 2. Raw vs fuel-corrected lap times
Fuel burn makes cars ~3 s faster over a race, which hides tyre wear. Correcting for it exposes the real degradation.

In [ ]:
drv = df["Driver"].value_counts().index[0]
d = df[df["Driver"] == drv]
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for c, g in d.groupby("Compound"):
    ax[0].scatter(g["LapNumber"], g["LapTimeS"], s=14, c=COLORS[c], label=c, edgecolor="k", lw=0.3)
    ax[1].scatter(g["LapNumber"], g["LapTimeCorr"], s=14, c=COLORS[c], label=c, edgecolor="k", lw=0.3)
ax[0].set(title=f"{drv}: raw lap time", xlabel="Lap", ylabel="s")
ax[1].set(title=f"{drv}: fuel-corrected", xlabel="Lap")
ax[0].legend(); plt.tight_layout(); plt.savefig("figures/fuel_correction.png", bbox_inches="tight")

## 3. Fit degradation models

In [ ]:
models = fit_all(df)
summary_table(models)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for c, m in models.items():
    g = df[df["Compound"] == c]
    ax.scatter(g["TyreLife"], g["Delta"], s=8, alpha=0.35, c=COLORS[c])
    a = np.arange(1, m.max_age_seen + 1)
    ax.plot(a, m.predict(a), c=COLORS[c], lw=2.5, label=f"{c} ({m.kind}, {m.deg_per_lap:.3f} s/lap)")
    if m.cliff_age:
        ax.axvline(m.cliff_age, c=COLORS[c], ls="--", lw=1)
ax.set(xlabel="Tyre age (laps)", ylabel="Lap-time delta to driver baseline (s)",
       title=f"Tyre degradation – {EVENT} {YEAR}")
ax.legend(); plt.savefig("figures/degradation_curves.png", bbox_inches="tight")

## 4. Sensitivity to the fuel assumption
The fuel correction is an assumption, so check how much the fitted deg rates move.

In [ ]:
rows = []
for spk in [0.025, 0.030, 0.035]:
    for fs in [90, 100, 110]:
        ms = fit_all(fuel_correct(clean, TOTAL_LAPS, fs, spk))
        rows.append({"s/kg": spk, "fuel_kg": fs, **{c: round(m.deg_per_lap, 4) for c, m in ms.items()}})
pd.DataFrame(rows)

## 5. Save models for the strategy notebook

In [ ]:
pit_loss = float("nan") if DEMO else pit_loss_from_session(laps_raw)
print("Estimated green-flag pit loss:", round(pit_loss, 2), "s")
tag = "demo" if DEMO else f"{YEAR}_{str(GP).lower().replace(' ', '_')}"
save_models(models, f"data/deg_{tag}.json",
            meta=dict(year=YEAR, gp=str(GP), event=EVENT, total_laps=TOTAL_LAPS,
                      pit_loss_est=pit_loss, fuel_start_kg=FUEL_START_KG, sec_per_kg=SEC_PER_KG))
print("saved", f"data/deg_{tag}.json")

## Findings
- Bahrain 2025 gave 1,129 race laps from OpenF1 (Medium 517, Hard 316, Soft 295). After cleaning, the linear fits gave 0.094 s/lap for the Soft, 0.082 s/lap for the Medium and 0.100 s/lap for the Hard.
- The Soft shows a possible cliff at around 21 laps. The longest Soft stint in the race was 28 laps.
- The pit loss estimated from in-laps and out-laps is 23.8 s, which matches the usual figure for Bahrain and suggests the lap cleaning is working.
- The Hard degrading faster than the Soft is not a real tyre effect. The Hard was mostly run in the second half of the race, and within a single stint tyre age and lap number increase together, so track evolution and any error in the fuel correction end up in the fitted slope. Notebook 03 fits each race as a whole and separates these effects.
- Limitations: traffic, tyre management by the drivers and track evolution are not modelled here, and the result comes from a single race.